In [14]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

The nvcc4jupyter extension is already loaded. To reload it, use:
  %reload_ext nvcc4jupyter


In [34]:
# 1. 清除舊的 libtorch (防止路徑衝突)
!rm -rf libtorch* # 2. 下載 CUDA 12.1 專用版 (這才是 T4 的真理)
!wget -O libtorch.zip https://download.pytorch.org/libtorch/cu121/libtorch-cxx11-abi-shared-with-deps-2.2.1%2Bcu121.zip
!unzip -q libtorch.zip

# 3. 測試 GPU 是否在系統層級可見
!nvidia-smi

--2026-03-24 23:10:44--  https://download.pytorch.org/libtorch/cu121/libtorch-cxx11-abi-shared-with-deps-2.2.1%2Bcu121.zip
Resolving download.pytorch.org (download.pytorch.org)... 18.238.238.114, 18.238.238.104, 18.238.238.82, ...
Connecting to download.pytorch.org (download.pytorch.org)|18.238.238.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2502645257 (2.3G) [application/zip]
Saving to: ‘libtorch.zip’

libtorch.zip        100%[===================>]   2.33G   214MB/s    in 22s     

2026-03-24 23:11:06 (109 MB/s) - ‘libtorch.zip’ saved [2502645257/2502645257]

Tue Mar 24 23:12:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |

In [61]:
%%writefile logos.cpp
#include <torch/torch.h>
#include <cuda_runtime_api.h> // 【升級】直接使用 NVIDIA 原生硬體 API
#include <iostream>
#include <vector>
#include <chrono>
#include <iomanip>
#include <numeric>

using namespace torch;

// --- 硬體級 VRAM 探針 ---
size_t get_physical_vram_used() {
    cudaDeviceSynchronize(); // 確保所有非同步的 GPU 運算都已排程並分配記憶體
    size_t free_byte;
    size_t total_byte;
    cudaMemGetInfo(&free_byte, &total_byte);
    return total_byte - free_byte;
}

class XOREntropyObserver : public nn::Module {
public:
    int L, W;
    Device device;
    nn::Linear root_embed{nullptr};
    nn::ModuleList network{nullptr};
    nn::ModuleList local_observers{nullptr};
    Tensor layer_attention;
    nn::Linear bp_output_layer{nullptr};

    XOREntropyObserver(int layers, int hidden_dim, Device dev)
        : L(layers), W(hidden_dim), device(dev) {
        root_embed = register_module("root_embed", nn::Linear(2, W));
        network = register_module("network", nn::ModuleList());
        local_observers = register_module("local_observers", nn::ModuleList());
        for (int i = 0; i < L; ++i) {
            auto seq = nn::Sequential(nn::Linear(W, W), nn::LayerNorm(nn::LayerNormOptions({W})));
            network->push_back(seq);
            local_observers->push_back(nn::Linear(W, 1));
        }
        layer_attention = register_parameter("layer_attention", torch::ones({L}) / (float)L);
        bp_output_layer = register_module("bp_output_layer", nn::Linear(W, 1));

        for (auto& p : this->parameters()) { if (p.dim() > 1) nn::init::kaiming_normal_(p); }
        this->to(device);
    }

    std::pair<Tensor, Tensor> generate_purified_xor(int sample_size) {
        std::vector<Tensor> x_list, y_list;
        while (x_list.size() < sample_size) {
            auto sample = torch::rand({1, 2}, device) * 4.0 - 2.0;
            if (torch::abs(sample[0][0]).item<float>() > 0.05 &&
                torch::abs(sample[0][1]).item<float>() > 0.05) {
                x_list.push_back(sample);
                y_list.push_back(torch::sign(sample.select(1,0) * sample.select(1,1)).view({1,1}));
            }
        }
        return {torch::cat(x_list, 0), torch::cat(y_list, 0)};
    }

    Tensor resonance_loss(Tensor pred, Tensor target) {
        auto smooth = torch::smooth_l1_loss(pred, target, torch::Reduction::None);
        auto res = 1.0 - torch::tanh(100.0 * target * pred);
        return torch::mean(smooth * res);
    }

    float evaluate(Tensor X, Tensor Y, bool is_logos) {
        NoGradGuard no_grad;
        auto root = root_embed->forward(X);
        auto cur = root;
        std::vector<Tensor> l_preds;
        for (int i = 0; i < L; ++i) {
            auto f = torch::abs(network->ptr<nn::SequentialImpl>(i)->forward(cur));
            cur = f + cur + root;
            if (is_logos) l_preds.push_back(local_observers->ptr<nn::LinearImpl>(i)->forward(cur));
        }
        Tensor final_out = is_logos ? torch::sum(torch::cat(l_preds, 1) * torch::softmax(layer_attention, 0), 1, true)
                                    : bp_output_layer->forward(cur);
        return ((torch::sign(final_out) == Y).sum().item<float>() / Y.size(0)) * 100.0;
    }

    // 回傳 pair: {執行時間(秒), 峰值動態記憶體(MB)}
    std::pair<double, double> train_engine(Tensor X, Tensor Y, int epochs, bool is_logos) {
        optim::Adam opt(this->parameters(), optim::AdamOptions(0.005));

        // 取得初始物理顯存佔用
        size_t base_memory = get_physical_vram_used();
        size_t peak_bytes = base_memory;

        auto start_time = std::chrono::high_resolution_clock::now();

        for (int e = 0; e < epochs; ++e) {
            opt.zero_grad();
            if (is_logos) {
                auto root = root_embed->forward(X);
                auto cur = root.detach();
                std::vector<Tensor> l_preds;
                for (int i = 0; i < L; ++i) {
                    auto next = torch::abs(network->ptr<nn::SequentialImpl>(i)->forward(cur)) + cur + root.detach();
                    auto lp = local_observers->ptr<nn::LinearImpl>(i)->forward(next);
                    auto loss = resonance_loss(lp, Y);

                    // 【物理探針 A】捕捉每一層算完後的顯存
                    size_t current_mem = get_physical_vram_used();
                    if (current_mem > peak_bytes) peak_bytes = current_mem;

                    loss.backward();
                    l_preds.push_back(lp.detach());
                    cur = next.detach();
                }
                resonance_loss(torch::sum(torch::cat(l_preds, 1) * torch::softmax(layer_attention, 0), 1, true), Y).backward();
            } else {
                auto root = root_embed->forward(X); auto cur = root;
                for (int i = 0; i < L; ++i) cur = torch::abs(network->ptr<nn::SequentialImpl>(i)->forward(cur)) + cur + root;
                auto loss = torch::mse_loss(bp_output_layer->forward(cur), Y);

                // 【物理探針 B】捕捉 10 層計算圖建立完畢的極限顯存
                size_t current_mem = get_physical_vram_used();
                if (current_mem > peak_bytes) peak_bytes = current_mem;

                loss.backward();
            }
            opt.step();
        }
        auto end_time = std::chrono::high_resolution_clock::now();
        double time_taken = std::chrono::duration<double>(end_time - start_time).count();

        // 計算純動態增長的 MB
        double peak_mb = 0.0;
        if (peak_bytes > base_memory) {
            peak_mb = (peak_bytes - base_memory) / (1024.0 * 1024.0);
        }
        return {time_taken, peak_mb};
    }
};

int main() {
    if (!torch::cuda::is_available()) return -1;
    Device dev(torch::kCUDA);

    int L = 10, W = 256, epochs = 200; // 稍微調降 Epoch 以加速測試
    int train_size = 2500, test_size = 2500, num_seeds = 10;

    std::vector<float> bp_accs, logos_accs;
    std::vector<double> bp_times, logos_times;
    std::vector<double> bp_mems, logos_mems;

    std::cout << "--- [Logos O(1) Memory vs Classical BP Memory] ---" << std::endl;
    std::cout << "Hardware Probe  : Native CUDA Memory Allocator" << std::endl;
    std::cout << "Network Depth   : " << L << " Layers" << std::endl;
    std::cout << "Data Size       : 2500 Train / 2500 Test" << std::endl;
    std::cout << "--------------------------------------------------" << std::endl;

    for (int seed = 1; seed <= num_seeds; ++seed) {
        torch::manual_seed(seed);
        XOREntropyObserver bp_obs(L, W, dev);
        auto data = bp_obs.generate_purified_xor(train_size);
        auto test = bp_obs.generate_purified_xor(test_size);

        auto bp_res = bp_obs.train_engine(data.first, data.second, epochs, false);
        bp_times.push_back(bp_res.first); bp_mems.push_back(bp_res.second);
        bp_accs.push_back(bp_obs.evaluate(test.first, test.second, false));

        torch::manual_seed(seed);
        XOREntropyObserver logos_obs(L, W, dev);
        auto logos_res = logos_obs.train_engine(data.first, data.second, epochs, true);
        logos_times.push_back(logos_res.first); logos_mems.push_back(logos_res.second);
        logos_accs.push_back(logos_obs.evaluate(test.first, test.second, true));

        std::cout << std::fixed << std::setprecision(2);
        std::cout << "Universe [" << std::setw(2) << seed << "] BP: " << bp_accs.back() << "% (" << bp_mems.back() << " MB) | Logos: " << logos_accs.back() << "% (" << logos_mems.back() << " MB)" << std::endl;
    }

    auto avg = [](auto& v) { return std::accumulate(v.begin(), v.end(), 0.0) / v.size(); };
    double avg_bp_mem = avg(bp_mems), avg_logos_mem = avg(logos_mems);

    std::cout << "\n=======================================================" << std::endl;
    std::cout << "          FINAL HARDWARE FOOTPRINT REPORT              " << std::endl;
    std::cout << "=======================================================" << std::endl;
    std::cout << "[1] Average Accuracy (N=" << num_seeds << ")" << std::endl;
    std::cout << "    - Classical BP : " << std::fixed << std::setprecision(3) << avg(bp_accs) << "%" << std::endl;
    std::cout << "    - Logos O(0)   : " << avg(logos_accs) << "%" << std::endl;

    std::cout << "\n[2] EMPIRICAL PEAK VRAM (COMPUTATIONAL GRAPH)" << std::endl;
    std::cout << "    - Classical BP : " << std::fixed << std::setprecision(2) << avg_bp_mem << " MB  <- O(L) Scaling" << std::endl;
    std::cout << "    - Logos O(0)   : " << avg_logos_mem << " MB  <- O(1) Constant!" << std::endl;

    // 防呆處理：避免除以零
    if (avg_logos_mem > 0) {
        std::cout << "    >> MEMORY SAVINGS: " << (avg_bp_mem / avg_logos_mem) << "x Less Hardware VRAM Used" << std::endl;
    } else {
        std::cout << "    >> MEMORY SAVINGS: Logos memory footprint was too small to register!" << std::endl;
    }
    std::cout << "=======================================================" << std::endl;

    return 0;
}

Overwriting logos.cpp


In [63]:
!g++ -O3 logos.cpp -o logos_engine \
    -D_GLIBCXX_USE_CXX11_ABI=1 \
    -I/content/libtorch/include \
    -I/content/libtorch/include/torch/csrc/api/include \
    -I/usr/local/cuda-12.8/include \
    -L/content/libtorch/lib \
    -L/usr/local/cuda-12.8/targets/x86_64-linux/lib \
    -Wl,-rpath,/content/libtorch/lib:/usr/local/cuda-12.8/targets/x86_64-linux/lib \
    -Wl,--no-as-needed -ltorch -ltorch_cpu -ltorch_cuda -lc10 -lcuda -lcudart

import os
os.environ['LD_LIBRARY_PATH'] = "/content/libtorch/lib:/usr/local/cuda-12.8/targets/x86_64-linux/lib:" + os.environ.get('LD_LIBRARY_PATH', '')
!./logos_engine

--- [Logos O(1) Memory vs Classical BP Memory] ---
Hardware Probe  : Native CUDA Memory Allocator
Network Depth   : 10 Layers
Data Size       : 2500 Train / 2500 Test
--------------------------------------------------
Universe [ 1] BP: 99.80% (144.00 MB) | Logos: 100.00% (6.00 MB)
Universe [ 2] BP: 100.00% (0.00 MB) | Logos: 99.80% (0.00 MB)
Universe [ 3] BP: 99.96% (0.00 MB) | Logos: 100.00% (0.00 MB)
Universe [ 4] BP: 99.96% (0.00 MB) | Logos: 100.00% (0.00 MB)
Universe [ 5] BP: 99.96% (0.00 MB) | Logos: 100.00% (0.00 MB)
Universe [ 6] BP: 100.00% (0.00 MB) | Logos: 100.00% (0.00 MB)
Universe [ 7] BP: 99.76% (0.00 MB) | Logos: 100.00% (0.00 MB)
Universe [ 8] BP: 99.88% (0.00 MB) | Logos: 100.00% (0.00 MB)
Universe [ 9] BP: 99.92% (0.00 MB) | Logos: 100.00% (0.00 MB)
Universe [10] BP: 100.00% (0.00 MB) | Logos: 100.00% (0.00 MB)

          FINAL HARDWARE FOOTPRINT REPORT              
[1] Average Accuracy (N=10)
    - Classical BP : 99.924%
    - Logos O(0)   : 99.980%

[2] EMPIRICAL 